# Stream Analytics — Milestone 2
## Food Delivery Platform: Real-Time Data Processing & Storage
**Group 5 · BBADBA A · iesstsabbadbaa-grp-01-05**



**Two feeds:**
| Topic | Schema | Description |
|---|---|---|
| `group_5_orders` | `OrderLifecycleEvent` | CREATED → ACCEPTED → PREP_STARTED → READY → PICKED_UP → DELIVERED / CANCELLED |
| `group_5_couriers` | `CourierStatusEvent` | ONLINE, OFFLINE, LOCATION, ASSIGNED, UNASSIGNED |



---
# Section 1: Configuration & Setup

## 1.1 — Connection configuration
Fill in your Event Hub and Azure Blob Storage credentials here. Run this cell first every session.

In [1]:
event_hub_namespace = 'iesstsabbadbaa-grp-01-05'

# Orders topic
orders_eventhub_name      = 'group_5_orders'
orders_producer_conn_str  = 'Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Producer;SharedAccessKey=WFBCkCoTsTCcbVM/wLyMfhbMDS65ErXd4+AEhFf3OEM=;EntityPath=group_5_orders'
orders_consumer_conn_str  = 'Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=TSrhH6ULsMonQ6K9irBC/xfp6XeErJqY4+AEhADkbSg=;EntityPath=group_5_orders'

# Couriers topic
couriers_eventhub_name     = 'group_5_couriers'
couriers_producer_conn_str = 'Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Producer;SharedAccessKey=y8NUju2lX9f2PPk+gdAoCi++Qpl2yaxbG+AEhBxbMFI=;EntityPath=group_5_couriers'
couriers_consumer_conn_str = 'Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=e+wqm3BUHKuQ7Bmrr+SYlEFjbzg7esWam+AEhFPyWnQ=;EntityPath=group_5_couriers'

# Azure Blob Storage
account_name    = 'iesstsabbadbaa'
account_key     = 'TZoXWmij7SjGyogaWJYIi179/BrsUcyWcPE5XleNTXI4Wqwak9JrkOSpNXzM88w39j5sjxupYtX6+AStInJJfQ=='
container_name  = 'group5'

# Blob output + checkpoint paths
orders_output_path       = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/orders/output/"
orders_checkpoint_path   = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/orders/checkpoint/"
couriers_output_path     = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/couriers/output/"
couriers_checkpoint_path = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/couriers/checkpoint/"

print("Config loaded.")
print(f"  Orders topic    : {orders_eventhub_name}")
print(f"  Couriers topic  : {couriers_eventhub_name}")
print(f"  Blob container  : {container_name}@{account_name}")

Config loaded.
  Orders topic    : group_5_orders
  Couriers topic  : group_5_couriers
  Blob container  : group5@iesstsabbadbaa


## 1.2 — Install Python dependencies

In [2]:
!pip install fastavro confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 108.5 MB/s eta 0:00:00


---
# Section 2: AVRO Schemas
Defined as JSON strings — same pattern as the reference notebook. These are the exact schemas from Milestone 1, unchanged.

## 2.1 — Order lifecycle event schema (`group_5_orders`)

In [3]:
# Define the AVRO schema as a string
order_schema = """
{
    "type": "record",
    "name": "OrderLifecycleEvent",
    "namespace": "food.delivery",
    "doc": "Order lifecycle events for real-time food delivery analytics",
    "fields": [
        {"name": "schema_version",         "type": "string",  "default": "1.0"},
        {"name": "event_id",               "type": "string"},
        {"name": "order_id",               "type": "string"},
        {"name": "event_type",             "type": {"type": "enum", "name": "OrderEventType",
                                                    "symbols": ["CREATED","ACCEPTED","PREP_STARTED",
                                                                "READY","PICKED_UP","DELIVERED","CANCELLED"]}},
        {"name": "event_time",             "type": "string"},
        {"name": "ingestion_time",         "type": "string"},
        {"name": "customer_id",            "type": "string"},
        {"name": "restaurant_id",          "type": "string"},
        {"name": "courier_id",             "type": ["null","string"], "default": null},
        {"name": "zone_id",                "type": "string"},
        {"name": "estimated_prep_min",     "type": ["null","int"],    "default": null},
        {"name": "estimated_delivery_min", "type": ["null","int"],    "default": null},
        {"name": "total_amount_eur",       "type": "float"},
        {"name": "promo_applied",          "type": "boolean"},
        {"name": "cancel_reason",          "type": ["null","string"], "default": null}
    ]
}
"""
print("Order schema defined.")

Order schema defined.


## 2.2 — Courier status event schema (`group_5_couriers`)

In [4]:
# Define the AVRO schema as a string
courier_schema = """
{
    "type": "record",
    "name": "CourierStatusEvent",
    "namespace": "food.delivery",
    "doc": "Courier availability + location + status events",
    "fields": [
        {"name": "schema_version",   "type": "string",  "default": "1.0"},
        {"name": "event_id",         "type": "string"},
        {"name": "courier_id",       "type": "string"},
        {"name": "event_type",       "type": {"type": "enum", "name": "CourierEventType",
                                              "symbols": ["ONLINE","OFFLINE","LOCATION","ASSIGNED","UNASSIGNED"]}},
        {"name": "event_time",       "type": "string"},
        {"name": "ingestion_time",   "type": "string"},
        {"name": "zone_id",          "type": "string"},
        {"name": "lat",              "type": "float"},
        {"name": "lon",              "type": "float"},
        {"name": "status",           "type": {"type": "enum", "name": "CourierStatus",
                                              "symbols": ["IDLE","EN_ROUTE_TO_RESTAURANT",
                                                          "WAITING","EN_ROUTE_TO_CUSTOMER","OFFLINE"]}},
        {"name": "current_order_id", "type": ["null","string"], "default": null},
        {"name": "battery_pct",      "type": "int"}
    ]
}
"""
print("Courier schema defined.")

Courier schema defined.


---
# Section 3: AVRO Producers
Both producers are written to disk using `%%writefile` then launched as background processes with `nohup`, following the same pattern as the reference notebook.

Each producer continuously generates synthetic food delivery events and sends them as AVRO-encoded bytes to Azure Event Hub.

## 3.1 — Write orders producer script to disk

In [5]:
%%writefile avro_producer_orders.py
#!/usr/bin/env python3

from confluent_kafka import Producer
import fastavro
from fastavro import parse_schema
import io, sys, time, random, datetime

schema = {
    "type": "record", "name": "OrderLifecycleEvent", "namespace": "food.delivery",
    "fields": [
        {"name": "schema_version",         "type": "string",  "default": "1.0"},
        {"name": "event_id",               "type": "string"},
        {"name": "order_id",               "type": "string"},
        {"name": "event_type",             "type": {"type": "enum", "name": "OrderEventType",
                                                    "symbols": ["CREATED","ACCEPTED","PREP_STARTED",
                                                                "READY","PICKED_UP","DELIVERED","CANCELLED"]}},
        {"name": "event_time",             "type": "string"},
        {"name": "ingestion_time",         "type": "string"},
        {"name": "customer_id",            "type": "string"},
        {"name": "restaurant_id",          "type": "string"},
        {"name": "courier_id",             "type": ["null","string"], "default": None},
        {"name": "zone_id",                "type": "string"},
        {"name": "estimated_prep_min",     "type": ["null","int"],    "default": None},
        {"name": "estimated_delivery_min", "type": ["null","int"],    "default": None},
        {"name": "total_amount_eur",       "type": "float"},
        {"name": "promo_applied",          "type": "boolean"},
        {"name": "cancel_reason",          "type": ["null","string"], "default": None}
    ]
}
parsed_schema = parse_schema(schema)

# Argument parsing — same pattern as reference notebook
print(sys.argv)
if len(sys.argv) < 4:
    print("Usage: python avro_producer_orders.py <event_hub_namespace> <eventhub_name> <eventhub_connection_string>")
    sys.exit(1)

event_hub_namespace     = sys.argv[1]
eventhub_name           = sys.argv[2]
eventhub_connection_str = sys.argv[3]

# Kafka (Event Hub Kafka mode) configuration — same as reference notebook
conf = {
    'bootstrap.servers': f"{event_hub_namespace}.servicebus.windows.net:9093",
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     '$ConnectionString',
    'sasl.password':     eventhub_connection_str,
    'client.id':         'orders-producer-group5'
}
producer = Producer(**conf)

def delivery_report(err, msg):
    if err is not None:
        print(f"Delivery failed: {err}")
    else:
        print(f"Delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

# Data generation helpers — from Milestone 1
random.seed(42)
ZONES        = ["Z1_Center","Z2_North","Z3_South","Z4_East","Z5_West"]
ZONE_WEIGHTS = [0.40,0.18,0.15,0.14,0.13]
LUNCH_HOURS  = list(range(12,15))
DINNER_HOURS = list(range(19,23))
START_DATE   = datetime.datetime(2026,2,1)
END_DATE     = datetime.datetime(2026,3,1)
restaurant_ids = [f"R{str(i).zfill(4)}" for i in range(1,121)]
courier_ids    = [f"C{str(i).zfill(5)}" for i in range(1,301)]
customer_ids   = [f"U{str(i).zfill(5)}" for i in range(1,801)]

def iso(dt): return dt.replace(microsecond=0).isoformat()
def pick_zone(): return random.choices(ZONES, weights=ZONE_WEIGHTS, k=1)[0]
def rand_dt(s,e):
    delta=e-s; return s+datetime.timedelta(seconds=random.randint(0,int(delta.total_seconds())-1))
def peak_weighted_time():
    dt=rand_dt(START_DATE,END_DATE); h=dt.hour; w=1.0
    if h in LUNCH_HOURS: w*=2.3
    if h in DINNER_HOURS: w*=2.8
    if dt.weekday()>=5: w*=1.25
    if random.random()<min(1.0,w/3.5): return dt
    return peak_weighted_time()
def maybe_late(t):
    if random.random()<0.05: return t+datetime.timedelta(minutes=random.randint(2,25))
    return t+datetime.timedelta(seconds=random.randint(1,45))
def gen_event_id(): return f"ORD_{random.randint(1,10**12):012d}"
def gen_order_id(n): return f"O{str(n).zfill(8)}"

def avro_serialize(message):
    with io.BytesIO() as buf:
        fastavro.schemaless_writer(buf, parsed_schema, message)
        return buf.getvalue()

# Produce AVRO messages in a continuous loop
print("Orders producer started. Sending to:", eventhub_name)
order_counter = 1
while True:
    order_id = gen_order_id(order_counter); order_counter += 1
    zone     = pick_zone()
    cust     = random.choice(customer_ids)
    rest     = random.choice(restaurant_ids)
    courier  = random.choice(courier_ids)
    t0       = peak_weighted_time()
    surge    = random.random() < 0.08 and t0.hour in (LUNCH_HOURS + DINNER_HOURS)
    promo    = random.random() < 0.10
    est_prep     = random.randint(8,25) + (10 if surge else 0)
    est_delivery = random.randint(12,40) + (8 if surge else 0)
    will_cancel  = random.random() < (0.06 + (0.05 if surge else 0) + (-0.02 if promo else 0))

    steps = [("CREATED", t0, None, None)]
    t_accepted = t0 + datetime.timedelta(minutes=random.randint(1,6))
    if will_cancel:
        if random.random() < 0.55:
            steps.append(("ACCEPTED", t_accepted, courier, None))
        cancel_time = t0 + datetime.timedelta(minutes=random.randint(1,18))
        reason = random.choice(["CUSTOMER_CHANGED_MIND","RESTAURANT_TOO_BUSY","COURIER_UNAVAILABLE"])
        steps.append(("CANCELLED", cancel_time, None, reason))
    else:
        steps.append(("ACCEPTED",     t_accepted, courier, None))
        t_prep = t_accepted + datetime.timedelta(minutes=random.randint(0,4))
        steps.append(("PREP_STARTED", t_prep, courier, None))
        t_ready = t_prep + datetime.timedelta(minutes=est_prep)
        steps.append(("READY",        t_ready, courier, None))
        t_pickup = t_ready + datetime.timedelta(minutes=random.randint(2,12))
        steps.append(("PICKED_UP",    t_pickup, courier, None))
        t_delivered = t_pickup + datetime.timedelta(minutes=random.randint(10,35))
        steps.append(("DELIVERED",    t_delivered, courier, None))

    for (event_type, event_time, cour_id, c_reason) in steps:
        ingestion = maybe_late(event_time)
        record = {
            "schema_version": "1.0",
            "event_id": gen_event_id(),
            "order_id": order_id,
            "event_type": event_type,
            "event_time": iso(event_time),
            "ingestion_time": iso(ingestion),
            "customer_id": cust,
            "restaurant_id": rest,
            "courier_id": cour_id,
            "zone_id": zone,
            "estimated_prep_min": est_prep if event_type in ("CREATED","ACCEPTED") else None,
            "estimated_delivery_min": est_delivery if event_type in ("CREATED","ACCEPTED") else None,
            "total_amount_eur": round(random.uniform(8,55), 2),
            "promo_applied": promo,
            "cancel_reason": c_reason
        }
        print(record)
        producer.produce(topic=eventhub_name, value=avro_serialize(record), callback=delivery_report)
        producer.poll(0)
        time.sleep(0.01)

    time.sleep(0.2)

producer.flush()

Writing avro_producer_orders.py


## 3.2 — Write couriers producer script to disk

In [6]:
%%writefile avro_producer_couriers.py
#!/usr/bin/env python3

from confluent_kafka import Producer
import fastavro
from fastavro import parse_schema
import io, sys, time, random, datetime

schema = {
    "type": "record", "name": "CourierStatusEvent", "namespace": "food.delivery",
    "fields": [
        {"name": "schema_version",   "type": "string",  "default": "1.0"},
        {"name": "event_id",         "type": "string"},
        {"name": "courier_id",       "type": "string"},
        {"name": "event_type",       "type": {"type": "enum", "name": "CourierEventType",
                                              "symbols": ["ONLINE","OFFLINE","LOCATION","ASSIGNED","UNASSIGNED"]}},
        {"name": "event_time",       "type": "string"},
        {"name": "ingestion_time",   "type": "string"},
        {"name": "zone_id",          "type": "string"},
        {"name": "lat",              "type": "float"},
        {"name": "lon",              "type": "float"},
        {"name": "status",           "type": {"type": "enum", "name": "CourierStatus",
                                              "symbols": ["IDLE","EN_ROUTE_TO_RESTAURANT",
                                                          "WAITING","EN_ROUTE_TO_CUSTOMER","OFFLINE"]}},
        {"name": "current_order_id", "type": ["null","string"], "default": None},
        {"name": "battery_pct",      "type": "int"}
    ]
}
parsed_schema = parse_schema(schema)

print(sys.argv)
if len(sys.argv) < 4:
    print("Usage: python avro_producer_couriers.py <event_hub_namespace> <eventhub_name> <eventhub_connection_string>")
    sys.exit(1)

event_hub_namespace     = sys.argv[1]
eventhub_name           = sys.argv[2]
eventhub_connection_str = sys.argv[3]

conf = {
    'bootstrap.servers': f"{event_hub_namespace}.servicebus.windows.net:9093",
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     '$ConnectionString',
    'sasl.password':     eventhub_connection_str,
    'client.id':         'couriers-producer-group5'
}
producer = Producer(**conf)

def delivery_report(err, msg):
    if err is not None:
        print(f"Delivery failed: {err}")
    else:
        print(f"Delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

random.seed(99)
ZONES        = ["Z1_Center","Z2_North","Z3_South","Z4_East","Z5_West"]
ZONE_WEIGHTS = [0.40,0.18,0.15,0.14,0.13]
LUNCH_HOURS  = list(range(12,15))
DINNER_HOURS = list(range(19,23))
START_DATE   = datetime.datetime(2026,2,1)
END_DATE     = datetime.datetime(2026,3,1)
courier_ids  = [f"C{str(i).zfill(5)}" for i in range(1,301)]
order_ids    = [f"O{str(i).zfill(8)}" for i in range(1,1001)]

def iso(dt): return dt.replace(microsecond=0).isoformat()
def pick_zone(): return random.choices(ZONES, weights=ZONE_WEIGHTS, k=1)[0]
def rand_dt(s,e):
    delta=e-s; return s+datetime.timedelta(seconds=random.randint(0,int(delta.total_seconds())-1))
def peak_weighted_time():
    dt=rand_dt(START_DATE,END_DATE); h=dt.hour; w=1.0
    if h in LUNCH_HOURS: w*=2.3
    if h in DINNER_HOURS: w*=2.8
    if dt.weekday()>=5: w*=1.25
    if random.random()<min(1.0,w/3.5): return dt
    return peak_weighted_time()
def maybe_late(t):
    if random.random()<0.05: return t+datetime.timedelta(minutes=random.randint(2,25))
    return t+datetime.timedelta(seconds=random.randint(1,45))
def gen_event_id(): return f"CUR_{random.randint(1,10**12):012d}"
def zone_latlon(z):
    c={"Z1_Center":(40.4168,-3.7038),"Z2_North":(40.4780,-3.6880),
       "Z3_South":(40.3800,-3.7000),"Z4_East":(40.4200,-3.6000),"Z5_West":(40.4200,-3.8000)}
    return c[z]
def jitter(lat,lon,m=800):
    d=m/111000.0; return lat+random.uniform(-d,d), lon+random.uniform(-d,d)

def avro_serialize(message):
    with io.BytesIO() as buf:
        fastavro.schemaless_writer(buf, parsed_schema, message)
        return buf.getvalue()

courier_state = {
    cid: {"online": False, "zone": pick_zone(), "lat": 0.0, "lon": 0.0, "status": "OFFLINE", "order": None}
    for cid in courier_ids
}
for cid in courier_ids:
    z=courier_state[cid]["zone"]; lat,lon=zone_latlon(z)
    courier_state[cid]["lat"],courier_state[cid]["lon"]=jitter(lat,lon)

print("Couriers producer started. Sending to:", eventhub_name)
while True:
    cid  = random.choice(courier_ids)
    st   = courier_state[cid]
    t    = peak_weighted_time()
    roll = random.random()

    if not st["online"] and roll < 0.20:
        st["online"]=True; st["status"]="IDLE"; event_type="ONLINE"
    elif st["online"] and roll < 0.08:
        st["online"]=False; st["status"]="OFFLINE"; st["order"]=None; event_type="OFFLINE"
    elif st["online"] and st["order"] is None and random.random() < 0.10:
        st["order"]=random.choice(order_ids)
        st["status"]=random.choice(["EN_ROUTE_TO_RESTAURANT","WAITING"]); event_type="ASSIGNED"
    elif st["online"] and st["order"] is not None and random.random() < 0.08:
        st["order"]=None; st["status"]="IDLE"; event_type="UNASSIGNED"
    else:
        event_type="LOCATION"
        if st["online"]:
            if random.random()<0.04: st["zone"]=pick_zone()
            blat,blon=zone_latlon(st["zone"]); st["lat"],st["lon"]=jitter(blat,blon,1200)
            st["status"]="IDLE" if st["order"] is None else random.choice(
                ["EN_ROUTE_TO_RESTAURANT","WAITING","EN_ROUTE_TO_CUSTOMER"])
        else:
            st["status"]="OFFLINE"

    # Courier offline mid-delivery edge case (from Milestone 1)
    if st["online"] and st["order"] is not None and random.random() < 0.02:
        offline_time = t + datetime.timedelta(seconds=random.randint(5,50))
        ingestion    = maybe_late(offline_time)
        offline_record = {
            "schema_version": "1.0", "event_id": gen_event_id(), "courier_id": cid,
            "event_type": "OFFLINE", "event_time": iso(offline_time), "ingestion_time": iso(ingestion),
            "zone_id": st["zone"], "lat": float(st["lat"]), "lon": float(st["lon"]),
            "status": "OFFLINE", "current_order_id": st["order"], "battery_pct": random.randint(1,100)
        }
        producer.produce(topic=eventhub_name, value=avro_serialize(offline_record), callback=delivery_report)
        producer.poll(0)
        st["online"]=False; st["status"]="OFFLINE"; st["order"]=None

    ingestion = maybe_late(t)
    record = {
        "schema_version": "1.0",
        "event_id": gen_event_id(),
        "courier_id": cid,
        "event_type": event_type,
        "event_time": iso(t),
        "ingestion_time": iso(ingestion),
        "zone_id": st["zone"],
        "lat": float(st["lat"]),
        "lon": float(st["lon"]),
        "status": st["status"],
        "current_order_id": st["order"],
        "battery_pct": random.randint(1,100)
    }
    print(record)
    producer.produce(topic=eventhub_name, value=avro_serialize(record), callback=delivery_report)
    producer.poll(0)
    time.sleep(0.01)

producer.flush()

Writing avro_producer_couriers.py


## 3.3 — Launch both producers in the background
Uses `nohup` so producers keep running while Spark cells execute below.

In [7]:
import os
os.environ['EH_NAMESPACE']   = event_hub_namespace
os.environ['ORDERS_TOPIC']   = orders_eventhub_name
os.environ['ORDERS_CONN']    = orders_producer_conn_str
os.environ['COURIERS_TOPIC'] = couriers_eventhub_name
os.environ['COURIERS_CONN']  = couriers_producer_conn_str

!nohup python avro_producer_orders.py $EH_NAMESPACE $ORDERS_TOPIC "$ORDERS_CONN" > orders_producer.log &
!nohup python avro_producer_couriers.py $EH_NAMESPACE $COURIERS_TOPIC "$COURIERS_CONN" > couriers_producer.log &

# Make sure events are sent to EventHub — same pattern as reference notebook
!sleep 10
!tail -20 orders_producer.log

!sleep 5
!tail -20 couriers_producer.log

!ps -ef | grep avro_pro

nohup: redirecting stderr to stdout
nohup: redirecting stderr to stdout
Delivered to group_5_orders [0] at offset 49
{'schema_version': '1.0', 'event_id': 'ORD_143417367493', 'order_id': 'O00000037', 'event_type': 'DELIVERED', 'event_time': '2026-02-09T09:59:44', 'ingestion_time': '2026-02-09T10:00:27', 'customer_id': 'U00155', 'restaurant_id': 'R0008', 'courier_id': 'C00231', 'zone_id': 'Z2_North', 'estimated_prep_min': None, 'estimated_delivery_min': None, 'total_amount_eur': 35.97, 'promo_applied': False, 'cancel_reason': None}
Delivered to group_5_orders [3] at offset 52
{'schema_version': '1.0', 'event_id': 'ORD_082269863101', 'order_id': 'O00000038', 'event_type': 'CREATED', 'event_time': '2026-02-26T13:17:33', 'ingestion_time': '2026-02-26T13:17:42', 'customer_id': 'U00318', 'restaurant_id': 'R0072', 'courier_id': None, 'zone_id': 'Z5_West', 'estimated_prep_min': 8, 'estimated_delivery_min': 23, 'total_amount_eur': 21.9, 'promo_applied': False, 'cancel_reason': None}
Delivered t

---
# Section 4: Spark Setup
Installs Java 21 and the latest Spark 4.x binary. Identical installation pattern to the reference notebooks.

In [8]:
import os
import subprocess

# Fetch the latest Spark 4.x version — same as reference notebook
spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-4\\.[0-9]\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_version

'spark-4.2.0'

In [12]:
import os
import subprocess

# Fetch the latest Spark 4.1.x version
# curl -s https://downloads.apache.org/spark/ → Fetches the Spark download page.
# grep -o 'spark-4\.1\+\.[0-9]\+' → Extracts only versions that start with spark-4.1
# sort -V → Sorts the versions numerically.
# tail -1 → Selects the latest version.

spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-4\\.1\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_version

spark_release  = spark_version
hadoop_version = 'hadoop3'

import os, time
start = time.time()
os.environ['SPARK_RELEASE']  = spark_release
os.environ['HADOOP_VERSION'] = hadoop_version
os.environ["JAVA_HOME"]      = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"]     = f"/content/{spark_release}-bin-{hadoop_version}"

# Install Java21 and Spark 4.x.y
!apt-get update
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz
!tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz

# Install findspark
!pip install -q findspark

import findspark
findspark.init()

# Check the pyspark version
import pyspark
print(pyspark.__version__)

!echo "==== Java JDK: $JAVA_HOME ===="
!/usr/lib/jvm/java-21-openjdk-amd64/bin/java --version
!echo "==== SPARK Binaries: $PWD/${SPARK_RELEASE}-bin-${HADOOP_VERSION} ===="
!ls ${SPARK_RELEASE}-bin-${HADOOP_VERSION}/bin/
!echo "================================"
!${SPARK_RELEASE}-bin-${HADOOP_VERSION}/bin/spark-shell --version
!echo "==== Colab Working Directory ===="
!pwd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,241 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

---
# Section 5: Spark Session & Stream Readers
Creates the Spark session with Azure Blob Storage JARs, then sets up `readStream` consumers for both Event Hub topics.

## 5.1 — Create Spark session
Adds `hadoop-azure` and `azure-storage` JARs on top of the standard `kafka` + `avro` JARs from the reference notebook — required to write Parquet to Azure Blob Storage.

In [13]:
# JARs needed for Hadoop-compatible access in Azure Storage
jar_dependencies = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1",
    "org.apache.spark:spark-avro_2.13:4.1.1",
    "org.apache.hadoop:hadoop-azure:3.3.1",         # Hadoop Azure connector
    "com.microsoft.azure:azure-storage:8.6.6"        # Azure Blob SDK dependency
])

from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro

# Create a Spark session
spark = SparkSession \
    .builder \
    .appName("FoodDelivery_StreamAnalytics_Group5") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config("spark.jars.packages", jar_dependencies) \
    .config(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key) \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

## 5.2 — Kafka configuration for the orders topic
Kafka source will create a unique group id for each query automatically. The `groupIdPrefix` auto-generates consumer group IDs internally but these DO NOT appear explicitly as consumer groups within the Azure Portal under Event Hub Consumer Groups.

In [14]:
kafkaConf_orders = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    # Below settings required if kafka is secured (connecting to Azure Event Hubs):
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{orders_consumer_conn_str}";',

    "subscribe": orders_eventhub_name,   # subscribe to the entire topic
    "startingOffsets": "earliest",        # "latest", "earliest"
    "enable.auto.commit": "true ",
    "groupIdPrefix": "Stream_Analytics_Group5_",
    "auto.commit.interval.ms": "5000"
}

kafkaConf_orders

{'kafka.bootstrap.servers': 'iesstsabbadbaa-grp-01-05.servicebus.windows.net:9093',
 'kafka.sasl.mechanism': 'PLAIN',
 'kafka.security.protocol': 'SASL_SSL',
 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=TSrhH6ULsMonQ6K9irBC/xfp6XeErJqY4+AEhADkbSg=;EntityPath=group_5_orders";',
 'subscribe': 'group_5_orders',
 'startingOffsets': 'earliest',
 'enable.auto.commit': 'true ',
 'groupIdPrefix': 'Stream_Analytics_Group5_',
 'auto.commit.interval.ms': '5000'}

## 5.3 — Read & deserialize the orders stream

In [15]:
# Read from Event Hub using Kafka
orders_df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConf_orders)

orders_df = orders_df.load()   # Start reading data from the specified Kafka topic
orders_df.printSchema()

# Deserialize the AVRO messages from the value column
orders_df = orders_df.select(from_avro(orders_df.value, order_schema).alias("order"))

# Print the schema of the DataFrame
orders_df.printSchema()

# Now we flatten the AVRO record to put each field in a separate column:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

orders_df = orders_df.select(
    col("order.schema_version"),
    col("order.event_id"),
    col("order.order_id"),
    col("order.event_type"),
    col("order.event_time").cast(TimestampType()).alias("event_time"),
    col("order.ingestion_time").cast(TimestampType()).alias("ingestion_time"),
    col("order.customer_id"),
    col("order.restaurant_id"),
    col("order.courier_id"),
    col("order.zone_id"),
    col("order.estimated_prep_min"),
    col("order.estimated_delivery_min"),
    col("order.total_amount_eur"),
    col("order.promo_applied"),
    col("order.cancel_reason")
)

orders_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- order: struct (nullable = true)
 |    |-- schema_version: string (nullable = false)
 |    |-- event_id: string (nullable = false)
 |    |-- order_id: string (nullable = false)
 |    |-- event_type: string (nullable = false)
 |    |-- event_time: string (nullable = false)
 |    |-- ingestion_time: string (nullable = false)
 |    |-- customer_id: string (nullable = false)
 |    |-- restaurant_id: string (nullable = false)
 |    |-- courier_id: string (nullable = true)
 |    |-- zone_id: string (nullable = false)
 |    |-- estimated_prep_min: integer (nullable = true)
 |    |-- estimated_delivery_min: integer (nullable = true)
 |    |-- total_amount_eur: float (nullable = false)
 |    |-- promo_applie

## 5.4 — Kafka configuration for the couriers topic

In [16]:
kafkaConf_couriers = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    # Below settings required if kafka is secured (connecting to Azure Event Hubs):
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{couriers_consumer_conn_str}";',

    "subscribe": couriers_eventhub_name,
    "startingOffsets": "earliest",
    "enable.auto.commit": "true ",
    "groupIdPrefix": "Stream_Analytics_Group5_",
    "auto.commit.interval.ms": "5000"
}

kafkaConf_couriers

{'kafka.bootstrap.servers': 'iesstsabbadbaa-grp-01-05.servicebus.windows.net:9093',
 'kafka.sasl.mechanism': 'PLAIN',
 'kafka.security.protocol': 'SASL_SSL',
 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=e+wqm3BUHKuQ7Bmrr+SYlEFjbzg7esWam+AEhFPyWnQ=;EntityPath=group_5_couriers";',
 'subscribe': 'group_5_couriers',
 'startingOffsets': 'earliest',
 'enable.auto.commit': 'true ',
 'groupIdPrefix': 'Stream_Analytics_Group5_',
 'auto.commit.interval.ms': '5000'}

## 5.5 — Read & deserialize the couriers stream

In [17]:
# Read from Event Hub using Kafka
couriers_df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConf_couriers)

couriers_df = couriers_df.load()   # Start reading data from the specified Kafka topic
couriers_df.printSchema()

# Deserialize the AVRO messages from the value column
couriers_df = couriers_df.select(from_avro(couriers_df.value, courier_schema).alias("courier"))

# Print the schema of the DataFrame
couriers_df.printSchema()

# Now we flatten the AVRO record to put each field in a separate column:
couriers_df = couriers_df.select(
    col("courier.schema_version"),
    col("courier.event_id"),
    col("courier.courier_id"),
    col("courier.event_type"),
    col("courier.event_time").cast(TimestampType()).alias("event_time"),
    col("courier.ingestion_time").cast(TimestampType()).alias("ingestion_time"),
    col("courier.zone_id"),
    col("courier.lat"),
    col("courier.lon"),
    col("courier.status"),
    col("courier.current_order_id"),
    col("courier.battery_pct")
)

couriers_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- courier: struct (nullable = true)
 |    |-- schema_version: string (nullable = false)
 |    |-- event_id: string (nullable = false)
 |    |-- courier_id: string (nullable = false)
 |    |-- event_type: string (nullable = false)
 |    |-- event_time: string (nullable = false)
 |    |-- ingestion_time: string (nullable = false)
 |    |-- zone_id: string (nullable = false)
 |    |-- lat: float (nullable = false)
 |    |-- lon: float (nullable = false)
 |    |-- status: string (nullable = false)
 |    |-- current_order_id: string (nullable = true)
 |    |-- battery_pct: integer (nullable = false)

root
 |-- schema_version: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- courier_id

---
# Section 6: Analytical Queries
Your Spark job and input messages are ready to be worked on. Each use case filters or aggregates the stream, writes to an in-memory table, then uses the polling loop pattern from the reference notebook to display results as they arrive.

| Use Case | Feed | Operation | Output mode |
|---|---|---|---|
| UC1 — Raw order monitor | orders | passthrough | update |
| UC2 — Cancellation monitor | orders | filter | update |
| UC3 — Revenue by zone | orders | groupBy + agg | complete |
| UC4 — Courier availability | couriers | groupBy + agg | complete |
| UC5 — Offline mid-delivery anomaly | couriers | filter | update |

In [ ]:
!mkdir checkpoint

## Use Case 1 — Raw order event monitor
Streams all order events into memory so we can inspect what is arriving live from Event Hub.

In [18]:
# Display all order records in the console
query_name = 'all_orders'
query = orders_df.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName(query_name) \
    .start()

    # .option("checkpointLocation", "checkpoint") \  # Checkpoint not valid for memory sink

spark.sql('show tables').show()

# Get the list of active streaming queries
active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

# Let Spark start fetching events from the EventHub Instance
!sleep 20

from IPython.display import display, clear_output
numTries = 1  # if None, iterate forever
i = 0
query_name = 'all_orders'

while not numTries or i < numTries:
    i += 1
    query_status = query.status.get('message')
    while query_status == 'Processing new data' or query_status == 'Waiting for data to arrive' or query_status == 'Waiting for next trigger':
        query_status = query.status.get('message')
        print(query.status)
        print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))
        time.sleep(1)
        clear_output(wait=True)
    query_status = query.status.get('message')
    print(query_status)
    print(spark.sql(f'SELECT * FROM {query_name}').show(20, truncate=True))
    print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))

Writing offsets to log
+--------------+----------------+---------+------------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|schema_version|        event_id| order_id|  event_type|         event_time|     ingestion_time|customer_id|restaurant_id|courier_id|  zone_id|estimated_prep_min|estimated_delivery_min|total_amount_eur|promo_applied|cancel_reason|
+--------------+----------------+---------+------------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|           1.0|ORD_752562516775|O00000002|   PICKED_UP|2026-02-18 13:40:07|2026-02-18 13:40:43|     U00388|        R0011|    C00283| Z2_North|              NULL|                  NULL|           23.24|         true|         NULL|
|           1.0|ORD_902926886138|O00000002|   DELIVER

## Use Case 2 — Cancellation monitor
Filters cancelled orders in real time. Helps detect cancellation spikes during surge periods.

In [19]:
# Filter to cancelled orders only
cancelled_df = orders_df.filter(orders_df.event_type == "CANCELLED")

query_name = 'cancelled_orders'
query = cancelled_df.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName(query_name) \
    .start()

    # .option("checkpointLocation", "checkpoint") \  # Checkpoint not valid for memory sink

spark.sql('show tables').show()

active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

!sleep 20

numTries = 1
i = 0
query_name = 'cancelled_orders'

while not numTries or i < numTries:
    i += 1
    query_status = query.status.get('message')
    while query_status == 'Processing new data' or query_status == 'Waiting for data to arrive' or query_status == 'Waiting for next trigger':
        query_status = query.status.get('message')
        print(query.status)
        print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))
        time.sleep(1)
        clear_output(wait=True)
    query_status = query.status.get('message')
    print(query_status)
    print(spark.sql(f'SELECT zone_id, cancel_reason, count(*) as cancellations FROM {query_name} GROUP BY zone_id, cancel_reason ORDER BY cancellations DESC').show(20, truncate=True))
    print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))

Processing new data
+---------+--------------------+-------------+
|  zone_id|       cancel_reason|cancellations|
+---------+--------------------+-------------+
|Z1_Center|CUSTOMER_CHANGED_...|           24|
|Z1_Center| RESTAURANT_TOO_BUSY|           21|
|Z1_Center| COURIER_UNAVAILABLE|           19|
|  Z4_East|CUSTOMER_CHANGED_...|           16|
|  Z4_East| COURIER_UNAVAILABLE|           12|
| Z3_South| RESTAURANT_TOO_BUSY|           11|
|  Z4_East| RESTAURANT_TOO_BUSY|           10|
| Z2_North| RESTAURANT_TOO_BUSY|            9|
| Z3_South|CUSTOMER_CHANGED_...|            9|
|  Z5_West|CUSTOMER_CHANGED_...|            8|
|  Z5_West| RESTAURANT_TOO_BUSY|            8|
| Z2_North| COURIER_UNAVAILABLE|            7|
| Z2_North|CUSTOMER_CHANGED_...|            6|
| Z3_South| COURIER_UNAVAILABLE|            6|
|  Z5_West| COURIER_UNAVAILABLE|            5|
+---------+--------------------+-------------+

None
+------------+
|record_count|
+------------+
|         171|
+------------+

None


## Use Case 3 — Revenue by zone
Aggregates total revenue from DELIVERED events grouped by zone. Shows which zones generate the most revenue in real time.

In [20]:
from pyspark.sql.functions import sum as spark_sum, avg, count

# Filter to delivered orders and aggregate by zone
delivered_df = orders_df.filter(orders_df.event_type == "DELIVERED")

revenue_by_zone_df = delivered_df.groupBy("zone_id").agg(
    count("order_id").alias("delivered_orders"),
    spark_sum("total_amount_eur").alias("total_revenue_eur"),
    avg("total_amount_eur").alias("avg_order_value_eur")
)

query_name = 'revenue_by_zone'
query = revenue_by_zone_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName(query_name) \
    .start()

    # .option("checkpointLocation", "checkpoint") \  # Checkpoint not valid for memory sink

spark.sql('show tables').show()

active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

!sleep 20

numTries = 1
i = 0
query_name = 'revenue_by_zone'

while not numTries or i < numTries:
    i += 1
    query_status = query.status.get('message')
    while query_status == 'Processing new data' or query_status == 'Waiting for data to arrive' or query_status == 'Waiting for next trigger':
        query_status = query.status.get('message')
        print(query.status)
        print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))
        time.sleep(1)
        clear_output(wait=True)
    query_status = query.status.get('message')
    print(query_status)
    print(spark.sql(f'SELECT zone_id, delivered_orders, round(total_revenue_eur,2) as total_revenue_eur, round(avg_order_value_eur,2) as avg_order_value_eur FROM {query_name} ORDER BY total_revenue_eur DESC').show(20, truncate=True))
    print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))

Processing new data
+---------+----------------+-----------------+-------------------+
|  zone_id|delivered_orders|total_revenue_eur|avg_order_value_eur|
+---------+----------------+-----------------+-------------------+
|Z1_Center|            1067|         33898.08|              31.77|
| Z2_North|             441|         14166.61|              32.12|
| Z3_South|             382|         12015.15|              31.45|
|  Z5_West|             364|         11380.19|              31.26|
|  Z4_East|             353|         11124.09|              31.51|
+---------+----------------+-----------------+-------------------+

None
+------------+
|record_count|
+------------+
|           5|
+------------+

None


## Use Case 4 — Courier availability by zone
Counts couriers per status per zone in real time. Identifies zones with low courier coverage during peak hours.

In [21]:
# Aggregate courier counts by zone and status
courier_availability_df = couriers_df.groupBy("zone_id", "status").agg(
    count("courier_id").alias("courier_count")
)

query_name = 'courier_availability'
query = courier_availability_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName(query_name) \
    .start()

    # .option("checkpointLocation", "checkpoint") \  # Checkpoint not valid for memory sink

spark.sql('show tables').show()

active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

!sleep 20

numTries = 1
i = 0
query_name = 'courier_availability'

while not numTries or i < numTries:
    i += 1
    query_status = query.status.get('message')
    while query_status == 'Processing new data' or query_status == 'Waiting for data to arrive' or query_status == 'Waiting for next trigger':
        query_status = query.status.get('message')
        print(query.status)
        print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))
        time.sleep(1)
        clear_output(wait=True)
    query_status = query.status.get('message')
    print(query_status)
    print(spark.sql(f'SELECT zone_id, status, courier_count FROM {query_name} ORDER BY zone_id, courier_count DESC').show(20, truncate=True))
    print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))

Processing new data
+---------+--------------------+-------------+
|  zone_id|              status|courier_count|
+---------+--------------------+-------------+
|Z1_Center|                IDLE|        12908|
|Z1_Center|             OFFLINE|         9136|
|Z1_Center|EN_ROUTE_TO_RESTA...|         2567|
|Z1_Center|             WAITING|         2491|
|Z1_Center|EN_ROUTE_TO_CUSTOMER|         1889|
| Z2_North|                IDLE|         6612|
| Z2_North|             OFFLINE|         4602|
| Z2_North|             WAITING|         1241|
| Z2_North|EN_ROUTE_TO_RESTA...|         1234|
| Z2_North|EN_ROUTE_TO_CUSTOMER|          983|
| Z3_South|                IDLE|         5192|
| Z3_South|             OFFLINE|         3570|
| Z3_South|EN_ROUTE_TO_RESTA...|          990|
| Z3_South|             WAITING|          954|
| Z3_South|EN_ROUTE_TO_CUSTOMER|          765|
|  Z4_East|                IDLE|         4913|
|  Z4_East|             OFFLINE|         3395|
|  Z4_East|EN_ROUTE_TO_RESTA...|        

## Use Case 5 — Couriers going offline mid-delivery (anomaly detection)
Detects couriers that emit an `OFFLINE` event while still carrying an active order (`current_order_id` is not null). These orders need immediate manual reassignment.

In [22]:
# Filter: offline event with an active order attached
offline_mid_delivery_df = couriers_df.filter(
    (couriers_df.event_type == "OFFLINE") & couriers_df.current_order_id.isNotNull()
)

query_name = 'offline_mid_delivery'
query = offline_mid_delivery_df.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName(query_name) \
    .start()

    # .option("checkpointLocation", "checkpoint") \  # Checkpoint not valid for memory sink

spark.sql('show tables').show()

active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

!sleep 20

numTries = 1
i = 0
query_name = 'offline_mid_delivery'

while not numTries or i < numTries:
    i += 1
    query_status = query.status.get('message')
    while query_status == 'Processing new data' or query_status == 'Waiting for data to arrive' or query_status == 'Waiting for next trigger':
        query_status = query.status.get('message')
        print(query.status)
        print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))
        time.sleep(1)
        clear_output(wait=True)
    query_status = query.status.get('message')
    print(query_status)
    print(spark.sql(f'SELECT * FROM {query_name}').show(20, truncate=True))
    print(spark.sql(f'SELECT count(*) as record_count FROM {query_name}').show(20, truncate=True))

Processing new data
+--------------+----------------+----------+----------+-------------------+-------------------+---------+---------+----------+-------+----------------+-----------+
|schema_version|        event_id|courier_id|event_type|         event_time|     ingestion_time|  zone_id|      lat|       lon| status|current_order_id|battery_pct|
+--------------+----------------+----------+----------+-------------------+-------------------+---------+---------+----------+-------+----------------+-----------+
|           1.0|CUR_363201101563|    C00059|   OFFLINE|2026-02-25 02:59:50|2026-02-25 02:59:52|Z1_Center|40.423733|-3.7047474|OFFLINE|       O00000407|         52|
|           1.0|CUR_813939254312|    C00030|   OFFLINE|2026-02-16 13:50:46|2026-02-16 13:51:24|Z1_Center|  40.4105|-3.6937375|OFFLINE|       O00000744|          1|
|           1.0|CUR_368854304247|    C00116|   OFFLINE|2026-02-14 07:01:49|2026-02-14 07:02:00|  Z5_West| 40.41786|-3.7997994|OFFLINE|       O00000304|         

---
# Section 7: Kafka Topics to Parquet Files — Azure Blob Storage
Writes both streams to Parquet files in Azure Blob Storage using the same `writeStream` pattern as the reference notebook, but targeting `wasbs://` paths instead of a local filesystem.

## 7.1 — Orders stream → Azure Blob Storage

In [23]:
query_name = 'orders_to_blob'
query_orders_parquet = orders_df.writeStream \
    .format("parquet") \
    .option("checkpointLocation", orders_checkpoint_path) \
    .option("path", orders_output_path) \
    .queryName(query_name) \
    .trigger(processingTime='20 seconds') \
    .start()

!sleep 20

# Get the list of active streaming queries
active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

for progress in query_orders_parquet.recentProgress:
    print("==== Orders Blob Progress ====")
    print(f"Batch ID: {progress['batchId']}")
    print(f"Input rows: {progress['numInputRows']}")
    print(f"Processed time: {progress['timestamp']}")
    print(f"Sink: {progress['sink']}")
    print("===")

Query Name: all_orders
Query ID: e97df4a1-bc4b-468b-9371-e2b33a6258bb
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: cancelled_orders
Query ID: 597b69b4-6893-49f4-9f3b-bd73f748079f
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: offline_mid_delivery
Query ID: 58e4afd3-1a0c-4e8d-9609-6b6e44adca64
Query Status: {'message': 'Getting offsets from KafkaV2[Subscribe[group_5_couriers]]', 'isDataAvailable': False, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: orders_to_blob
Query ID: 7a601653-60a8-4cd5-a404-601aaa073cff
Query Status: {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
Is Query Active: True
------------------

## 7.2 — Couriers stream → Azure Blob Storage

In [24]:
query_name = 'couriers_to_blob'
query_couriers_parquet = couriers_df.writeStream \
    .format("parquet") \
    .option("checkpointLocation", couriers_checkpoint_path) \
    .option("path", couriers_output_path) \
    .queryName(query_name) \
    .trigger(processingTime='20 seconds') \
    .start()

!sleep 20

active_queries = spark.streams.active
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)

for progress in query_couriers_parquet.recentProgress:
    print("==== Couriers Blob Progress ====")
    print(f"Batch ID: {progress['batchId']}")
    print(f"Input rows: {progress['numInputRows']}")
    print(f"Processed time: {progress['timestamp']}")
    print(f"Sink: {progress['sink']}")
    print("===")

Query Name: all_orders
Query ID: e97df4a1-bc4b-468b-9371-e2b33a6258bb
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: couriers_to_blob
Query ID: b0b7546a-7fa1-4c55-a18f-8ef16bb123a1
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: cancelled_orders
Query ID: 597b69b4-6893-49f4-9f3b-bd73f748079f
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: offline_mid_delivery
Query ID: 58e4afd3-1a0c-4e8d-9609-6b6e44adca64
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name

---
# Section 8: Spark Warehouse — Read Parquet Back from Blob
Once data is at rest in Blob Storage it is readable by Spark batch jobs, DuckDB, PowerBI, or Streamlit dashboards.

In [25]:
# Read orders Parquet back from Blob
df_orders_parquet = spark.read.parquet(orders_output_path)
df_orders_parquet.show(5)

# Read couriers Parquet back from Blob
df_couriers_parquet = spark.read.parquet(couriers_output_path)
df_couriers_parquet.show(5)

+--------------+----------------+---------+------------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|schema_version|        event_id| order_id|  event_type|         event_time|     ingestion_time|customer_id|restaurant_id|courier_id|  zone_id|estimated_prep_min|estimated_delivery_min|total_amount_eur|promo_applied|cancel_reason|
+--------------+----------------+---------+------------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|           1.0|ORD_614294294452|O00000001|     CREATED|2026-02-12 21:19:10|2026-02-12 21:19:49|     U00026|        R0095|      NULL| Z3_South|                25|                    14|           17.35|        false|         NULL|
|           1.0|ORD_050711229824|O00000001|   DELIVERED|2026-02-12 22:04:10|

In [26]:
# Sample: delivered orders only
df_delivered = df_orders_parquet.filter("event_type = 'DELIVERED'")
df_delivered.show(5)
df_delivered_pd = df_delivered.toPandas()
len(df_delivered_pd)

+--------------+----------------+---------+----------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|schema_version|        event_id| order_id|event_type|         event_time|     ingestion_time|customer_id|restaurant_id|courier_id|  zone_id|estimated_prep_min|estimated_delivery_min|total_amount_eur|promo_applied|cancel_reason|
+--------------+----------------+---------+----------+-------------------+-------------------+-----------+-------------+----------+---------+------------------+----------------------+----------------+-------------+-------------+
|           1.0|ORD_050711229824|O00000001| DELIVERED|2026-02-12 22:04:10|2026-02-12 22:04:27|     U00026|        R0095|    C00141| Z3_South|              NULL|                  NULL|            42.3|        false|         NULL|
|           1.0|ORD_064290117191|O00000006| DELIVERED|2026-02-07 20:06:27|2026-02-07

3371

In [27]:
# Sample: anomaly detection from Blob
df_anomalies = df_couriers_parquet.filter(
    "event_type = 'OFFLINE' AND current_order_id IS NOT NULL"
)
df_anomalies.show(5)
df_anomalies_pd = df_anomalies.toPandas()
len(df_anomalies_pd)

+--------------+----------------+----------+----------+-------------------+-------------------+---------+---------+----------+-------+----------------+-----------+
|schema_version|        event_id|courier_id|event_type|         event_time|     ingestion_time|  zone_id|      lat|       lon| status|current_order_id|battery_pct|
+--------------+----------------+----------+----------+-------------------+-------------------+---------+---------+----------+-------+----------------+-----------+
|           1.0|CUR_697586702326|    C00259|   OFFLINE|2026-02-12 10:43:01|2026-02-12 10:43:16|Z1_Center| 40.41679|-3.7017353|OFFLINE|       O00000532|         65|
|           1.0|CUR_961177941740|    C00205|   OFFLINE|2026-02-05 22:10:01|2026-02-05 22:10:17| Z2_North|40.482456|-3.6849837|OFFLINE|       O00000201|         69|
|           1.0|CUR_977064032484|    C00071|   OFFLINE|2026-02-01 06:06:35|2026-02-01 06:07:13| Z3_South|40.369705|-3.7018702|OFFLINE|       O00000913|         12|
|           1.0|

437

---
# Section 9: DuckDB — Batch SQL on Parquet Files
[DuckDB](https://duckdb.org/) is a modern analytical database that natively reads Parquet files. This demonstrates how the data at rest can be queried outside of Spark — for dashboards, reports, or ad-hoc analysis.

In [28]:
!pip install -q duckdb
import duckdb

# Save local copies so DuckDB can read from the local filesystem
local_orders_path   = "/content/local_orders/"
local_couriers_path = "/content/local_couriers/"

df_orders_parquet.write.mode("overwrite").parquet(local_orders_path)
df_couriers_parquet.write.mode("overwrite").parquet(local_couriers_path)

In [29]:
duckdb.sql(f"""
    SELECT zone_id,
           count(*) as delivered_orders,
           round(sum(total_amount_eur), 2) as total_revenue_eur,
           round(avg(total_amount_eur), 2) as avg_order_value_eur
    FROM read_parquet('{local_orders_path}*.parquet', hive_partitioning=1)
    WHERE event_type = 'DELIVERED'
    GROUP BY zone_id
    ORDER BY total_revenue_eur DESC
""")

┌───────────┬──────────────────┬───────────────────┬─────────────────────┐
│  zone_id  │ delivered_orders │ total_revenue_eur │ avg_order_value_eur │
│  varchar  │      int64       │      double       │       double        │
├───────────┼──────────────────┼───────────────────┼─────────────────────┤
│ Z1_Center │             1359 │           43021.4 │               31.66 │
│ Z2_North  │              586 │          18277.69 │               31.19 │
│ Z3_South  │              502 │          15644.97 │               31.17 │
│ Z5_West   │              464 │          14725.46 │               31.74 │
│ Z4_East   │              460 │          14679.78 │               31.91 │
└───────────┴──────────────────┴───────────────────┴─────────────────────┘

In [30]:
duckdb.sql(f"""
    SELECT courier_id, current_order_id, zone_id, battery_pct, event_time
    FROM read_parquet('{local_couriers_path}*.parquet', hive_partitioning=1)
    WHERE event_type = 'OFFLINE' AND current_order_id IS NOT NULL
    ORDER BY event_time DESC
    LIMIT 10
""")

┌────────────┬──────────────────┬───────────┬─────────────┬─────────────────────┐
│ courier_id │ current_order_id │  zone_id  │ battery_pct │     event_time      │
│  varchar   │     varchar      │  varchar  │    int32    │      timestamp      │
├────────────┼──────────────────┼───────────┼─────────────┼─────────────────────┤
│ C00050     │ O00000976        │ Z1_Center │          66 │ 2026-02-28 20:34:18 │
│ C00108     │ O00000679        │ Z1_Center │          27 │ 2026-02-28 20:16:00 │
│ C00094     │ O00000110        │ Z1_Center │          49 │ 2026-02-28 19:49:14 │
│ C00216     │ O00000006        │ Z5_West   │           8 │ 2026-02-28 19:40:24 │
│ C00290     │ O00000735        │ Z1_Center │          92 │ 2026-02-28 18:10:27 │
│ C00189     │ O00000543        │ Z3_South  │          54 │ 2026-02-28 14:58:07 │
│ C00093     │ O00000990        │ Z1_Center │          63 │ 2026-02-28 12:55:53 │
│ C00136     │ O00000397        │ Z2_North  │          13 │ 2026-02-28 12:20:10 │
│ C00100     │ O

---
# Section 10: Stop All Queries
Set the `if False` guard to `if True` and run this cell to cleanly stop all active streaming queries and the Spark session.

In [31]:
# Set to True and run cell when you want to stop your queries and Spark job.
if False:
    # Get the list of active streaming queries
    active_queries = spark.streams.active

    # Stop each active query
    for query in active_queries:
        query.stop()
        print(f"Query Name: {query.name}")
        print(f"Query ID: {query.id}")
        print(f"Query Status: {query.status}")
        print(f"Is Query Active: {query.isActive}")
        print("-" * 50)

    # Stop producers
    !pkill -f avro_producer_orders.py
    !pkill -f avro_producer_couriers.py

    spark.stop()
    spark.sparkContext.stop()

In [56]:
!pip install streamlit azure-storage-blob pyarrow plotly pandas
